In [ ]:
# ============================================================
# ASSIGNMENT 2 - A2_03c: NRMS TITLE+ABSTRACT (MIND, LARGE) - Q3 Improvement 2 (neural)
# Team: Shivam Patel + Suyash Pande | CS4.406 IRE
#
# Principled improvement over the title-only NRMS baseline (dev AUC 0.585):
# richer news encoding using TITLE + ABSTRACT, trained properly with lr=1e-4 + grad clip
# (the title-only baseline diverged at lr=1e-3). Same architecture otherwise.
#
# HF donbosoc/mind-artifacts: DOWNLOAD without token (public), UPLOAD with HF_TOKEN.
# Per-epoch checkpoint + HF push + early stop + divergence guard.
# Saves: mind_nrms_ta_ep*.pt, mind_nrms_ta_best.pt, mind_nrms_ta_devscores.pkl
# NEEDS: Kaggle MIND-large mounted.
# ============================================================
!pip install huggingface_hub -q
import os, glob, re, numpy as np, pandas as pd, random, time, pickle
import torch, torch.nn as nn, torch.nn.functional as Fnn
from huggingface_hub import hf_hub_download, HfApi
random.seed(0); torch.manual_seed(0)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:",device)
HF_REPO="donbosoc/mind-artifacts"

WRITE_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    WRITE_TOKEN=UserSecretsClient().get_secret("HF_TOKEN"); print("write token loaded (uploads enabled)")
except Exception as e:
    print("no HF_TOKEN -> downloads still work; uploads skipped:", e)
def dl(name): return hf_hub_download(HF_REPO, name, repo_type="dataset")   # public read
def push(local,name):
    if WRITE_TOKEN is None: print("  skip upload:",name); return
    try:
        HfApi(token=WRITE_TOKEN).upload_file(path_or_fileobj=local,path_in_repo=name,repo_id=HF_REPO,repo_type="dataset")
        print("  pushed",name)
    except Exception as e: print("  upload failed",name,"->",e)

LTR=glob.glob("/kaggle/input/**/MINDlarge_train",recursive=True)[0]
LDV=glob.glob("/kaggle/input/**/MINDlarge_dev",recursive=True)[0]

# ---- config ----
LR=1e-4; GRAD_CLIP=1.0; EPOCHS=2; BATCH=64; NPRATIO=4
TRAIN_CAP=800_000   # sample cap for speed (2 epochs over 800k)
MAX_LEN=50           # title+abstract -> longer sequence than title-only (was 20)
MAX_HIST=30; EARLY_STOP_PATIENCE=1


In [ ]:
# ---- vocab over TITLE + ABSTRACT ----
NEWS=["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH =["impression_id","user_id","time","history","impressions"]
_WORD=re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"
news=pd.concat([pd.read_csv(f"{d}/news.tsv",sep="\t",header=None,names=NEWS,quoting=3,
               usecols=["news_id","title","abstract"]) for d in (LTR,LDV)]).drop_duplicates("news_id").reset_index(drop=True)
news["title"]=news["title"].fillna(""); news["abstract"]=news["abstract"].fillna("")
# KEY CHANGE: text = title + abstract
text_tokens={pfx(r.news_id): tok(f"{r.title} {r.abstract}")[:MAX_LEN] for r in news.itertuples()}
vocab={"<pad>":0,"<unk>":1}
for tks in text_tokens.values():
    for w in tks:
        if w not in vocab: vocab[w]=len(vocab)
art_ids=[pfx(r.news_id) for r in news.itertuples()]; art_row={a:i for i,a in enumerate(art_ids)}
def enc(aid):
    idx=[vocab.get(w,1) for w in text_tokens.get(aid,[])][:MAX_LEN]
    return idx+[0]*(MAX_LEN-len(idx))
text_idx=np.array([enc(a) for a in art_ids],dtype=np.int64)
print("vocab:",len(vocab),"| text_idx (title+abstract):",text_idx.shape)


In [ ]:
# ---- NRMS (same architecture; now over longer title+abstract sequences) ----
class AdditiveAttention(nn.Module):
    def __init__(s,dim,hid=200):
        super().__init__(); s.p=nn.Linear(dim,hid); s.q=nn.Linear(hid,1,bias=False)
    def forward(s,x,mask=None):
        e=torch.tanh(s.p(x)); a=s.q(e).squeeze(-1)
        if mask is not None: a=a.masked_fill(mask==0,-1e9)
        return (x*torch.softmax(a,-1).unsqueeze(-1)).sum(1)
class NewsEncoder(nn.Module):
    def __init__(s,V,ed=300,h=15,hd=20):
        super().__init__(); d=h*hd
        s.emb=nn.Embedding(V,ed,padding_idx=0); s.attn=nn.MultiheadAttention(ed,h,batch_first=True)
        s.proj=nn.Linear(ed,d); s.add=AdditiveAttention(d); s.drop=nn.Dropout(0.2)
    def forward(s,t):
        m=(t!=0); x=s.drop(s.emb(t)); x,_=s.attn(x,x,x,key_padding_mask=~m)
        x=s.drop(x); x=torch.relu(s.proj(x)); return s.add(x,m)
class UserEncoder(nn.Module):
    def __init__(s,d):
        super().__init__(); s.attn=nn.MultiheadAttention(d,5,batch_first=True); s.add=AdditiveAttention(d)
    def forward(s,hv,mask):
        x,_=s.attn(hv,hv,hv,key_padding_mask=~mask); return s.add(x,mask)
class NRMS(nn.Module):
    def __init__(s,V,h=15,hd=20):
        super().__init__(); s.news=NewsEncoder(V,h=h,hd=hd); s.user=UserEncoder(h*hd)
    def forward(s,ht,hm,ct):
        B,H,L=ht.shape; C=ct.shape[1]
        hv=s.news(ht.reshape(B*H,L)).reshape(B,H,-1); u=s.user(hv,hm)
        cv=s.news(ct.reshape(B*C,L)).reshape(B,C,-1)
        return torch.bmm(cv,u.unsqueeze(-1)).squeeze(-1)
model=NRMS(len(vocab)).to(device)
print("NRMS(title+abstract) params:", sum(p.numel() for p in model.parameters()))


In [ ]:
# ---- samples + dev eval ----
def load_beh(p):
    b=pd.read_csv(f"{p}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3); return b
b_tr=load_beh(LTR); b_dv=load_beh(LDV)
hist_lut={}
for b in (b_tr,b_dv):
    for u,h in zip(b["user_id"],b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)]=[pfx(x) for x in h.split()]
def make_samples(b_df):
    S=[]
    for u,imps in zip(b_df["user_id"],b_df["impressions"]):
        if not isinstance(imps,str): continue
        uid=pfx(u); hist=hist_lut.get(uid)
        if not hist: continue
        pos=[pfx(t.split("-")[0]) for t in imps.split() if t.endswith("-1")]
        neg=[pfx(t.split("-")[0]) for t in imps.split() if t.endswith("-0")]
        if not pos or not neg: continue
        for p in pos:
            ng=random.sample(neg,NPRATIO) if len(neg)>=NPRATIO else neg+[random.choice(neg) for _ in range(NPRATIO-len(neg))]
            S.append((uid,p,ng))
        if len(S)>=TRAIN_CAP: break
    return S
train_samples=make_samples(b_tr)
random.Random(0).shuffle(train_samples); train_samples=train_samples[:TRAIN_CAP]
print("train samples (capped):",len(train_samples))
def hist_tensor(uid):
    h=hist_lut.get(uid,[])[-MAX_HIST:]
    rows=[art_row[a] for a in h if a in art_row][:MAX_HIST]
    mask=[1]*len(rows)+[0]*(MAX_HIST-len(rows)); rows=rows+[0]*(MAX_HIST-len(rows))
    return rows,mask
def collate(batch):
    H=[];HM=[];C=[]
    for uid,pos,negs in batch:
        hr,hm=hist_tensor(uid); H.append(hr);HM.append(hm); C.append([art_row.get(c,0) for c in [pos]+negs])
    H=np.array(H);C=np.array(C)
    return (torch.tensor(text_idx[H]),torch.tensor(np.array(HM),dtype=torch.bool),
            torch.tensor(text_idx[C]),torch.zeros(len(batch),dtype=torch.long))
def _auc(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def _mrr(s,lb):
    for rank,idx in enumerate(np.argsort(-s),1):
        if lb[idx]==1: return 1.0/rank
    return 0.0
def _ndcg(s,lb,k):
    o=np.argsort(-s)[:k];g=lb[o]
    dcg=sum(gg/np.log2(i+2) for i,gg in enumerate(g))
    idcg=sum(gg/np.log2(i+2) for i,gg in enumerate(np.sort(lb)[::-1][:k]))
    return float(dcg/idcg) if idcg>0 else 0.0
dev_impr=[]
for iid,u,imps in zip(b_dv["impression_id"],b_dv["user_id"],b_dv["impressions"]):
    if not isinstance(imps,str): continue
    uid=pfx(u)
    if not hist_lut.get(uid): continue
    cand=[pfx(t.split("-")[0]) for t in imps.split()]
    labs=np.array([1 if t.endswith("-1") else 0 for t in imps.split()])
    if labs.sum()>0: dev_impr.append((iid,uid,cand,labs))
random.Random(0).shuffle(dev_impr); dev_impr=dev_impr[:40000]
print("dev eval set:",len(dev_impr))
@torch.no_grad()
def eval_nrms(cache=None):
    model.eval(); A=[];M=[];N5=[];N10=[]
    for iid,uid,cand,labs in dev_impr:
        hr,hm=hist_tensor(uid)
        ht=torch.tensor(text_idx[np.array(hr)]).unsqueeze(0).to(device)
        hmask=torch.tensor(hm,dtype=torch.bool).unsqueeze(0).to(device)
        ct=torch.tensor(text_idx[np.array([art_row.get(c,0) for c in cand])]).unsqueeze(0).to(device)
        s=model(ht,hmask,ct).squeeze(0).cpu().numpy()
        if cache is not None:
            for c,sc in zip(cand,s): cache[(iid,c)]=float(sc)
        a=_auc(s,labs)
        if a is not None: A.append(a)
        M.append(_mrr(s,labs));N5.append(_ndcg(s,labs,5));N10.append(_ndcg(s,labs,10))
    return dict(AUC=np.mean(A),MRR=np.mean(M),nDCG5=np.mean(N5),nDCG10=np.mean(N10))


In [ ]:
# ---- train with lr=1e-4 + grad clip + divergence guard + per-epoch push ----
from torch.utils.data import DataLoader
opt=torch.optim.Adam(model.parameters(),lr=LR)
loader=DataLoader(train_samples,batch_size=BATCH,shuffle=True,collate_fn=collate,num_workers=2)
def save(tag):
    p=f"/kaggle/working/mind_nrms_ta_{tag}.pt"; torch.save({"model":model.state_dict(),"vocab":vocab},p); push(p,f"mind_nrms_ta_{tag}.pt")
best=-1; best_ep=-1; patience=0
for ep in range(EPOCHS):
    model.train(); t0=time.time(); tot=0; nb=0; run=[]
    for ht,hm,ct,y in loader:
        ht,hm,ct,y=ht.to(device),hm.to(device),ct.to(device),y.to(device)
        loss=Fnn.cross_entropy(model(ht,hm,ct),y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); opt.step()
        tot+=loss.item(); nb+=1; run.append(loss.item())
        if nb==200:
            dt=time.time()-t0; print(f"  [ep{ep}] 200 steps {dt:.1f}s -> ~{dt/200*len(loader)/60:.1f} min/epoch")
        if nb%2000==0:
            recent=np.mean(run[-2000:]); start=np.mean(run[:2000])
            print(f"  ep{ep} step {nb}/{len(loader)} loss {tot/nb:.4f} (recent2k {recent:.4f})")
            if nb>=10000 and recent>start+0.02:
                print(f"  DIVERGENCE GUARD: recent {recent:.4f} > start {start:.4f}+0.02 -> stop epoch"); break
    m=eval_nrms()
    print(f"epoch {ep}: loss {tot/max(1,nb):.4f} | dev AUC {m['AUC']:.4f} MRR {m['MRR']:.4f} nDCG@10 {m['nDCG10']:.4f} ({(time.time()-t0)/60:.1f} min)")
    save(f"ep{ep}")
    if m["AUC"]>best: best=m["AUC"]; best_ep=ep; save("best")
    else:
        patience+=1
        if patience>EARLY_STOP_PATIENCE: print(f"early stop (best {best:.4f} @ ep{best_ep})"); break
print(f"NRMS(title+abstract) best dev AUC {best:.4f} @ epoch {best_ep}")


In [ ]:
# ---- load best, final eval + cache dev scores ----
ck=torch.load(dl("mind_nrms_ta_best.pt"), map_location=device)   # public read
model.load_state_dict(ck["model"])
cache={}; m=eval_nrms(cache=cache)
print("=== NRMS(title+abstract) best dev ===", {k:round(v,4) for k,v in m.items()})
pickle.dump(cache, open("/kaggle/working/mind_nrms_ta_devscores.pkl","wb"))
push("/kaggle/working/mind_nrms_ta_devscores.pkl","mind_nrms_ta_devscores.pkl")
print("\nCompare to title-only NRMS baseline (0.5846). Improvement 2 = title+abstract.")
